# 09.01 · Why (and when not) to go multi-agent

**Module 09 · Multi-Agent Systems** · Part 2 (Pro)

| | |
|---|---|
| ⏱️ Estimated time | 90–120 min |
| 📋 Prerequisites | Part 1 (Modules 02, 04, 07, 08). Read the module README's **Welcome to Part 2** first. |
| 🔑 Keys | none (optional: `GROQ_API_KEY` for the 💸 cell) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Name** the five multi-agent patterns in the docs (subagents, handoffs, skills, router, custom workflow) and how control flows in each.
2. **Measure** patterns by model calls, tokens and latency on the same system, and compare with the docs' table.
3. **Decide** when a single agent with good tools is the better design, using a checklist.
4. **Place** LangChain (framework), LangGraph (runtime) and Deep Agents (harness).
5. **Organize** a long notebook into Parts and pair it with a `.py` file using Jupytext (this module's Jupyter skill).

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Multi-agent](https://docs.langchain.com/oss/python/langchain/multi-agent/index) | The five patterns, "when to use multi-agent", the model-call and token comparison tables |
| [Runtimes, frameworks, and harnesses](https://docs.langchain.com/oss/python/concepts/products) | LangGraph (runtime) vs LangChain (framework) vs Deep Agents (harness) |
| [Migrate from langgraph-supervisor](https://docs.langchain.com/oss/python/migrate/langgraph-supervisor) | Why `create_supervisor` is gone from new code (no longer actively maintained) |
| [Jupytext: paired notebooks](https://jupytext.readthedocs.io/en/latest/paired-notebooks.html) | `--set-formats`, `--sync`, the percent format |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-groq>=1.1.3,<2" "langgraph>=1.2.14,<2" "jupytext>=1.19.6,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langgraph", "jupytext",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/09-multi-agent-systems/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 09 LIVE FAQ index): no usage telemetry.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# fastembed (Project 09 LIVE FAQ index) imports tqdm.auto; without ipywidgets it warns once. Progress bars still work.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

> Everything below runs 🧪 offline on the **Orbita Cloud support desk** (`mastery.agents.support`), the domain of the whole module. Its offline model is rule-based: it reads its role from the system prompt and its options from the tools it was given, so it behaves like a very literal tool-calling LLM and its model-call counts are exact.

---
## Part 1 · Five patterns, one question: who decides what happens next?

| Pattern | How it works | Who controls the flow |
|---|---|---|
| **Subagents** | A main agent calls specialist agents **as tools**; results come back to it | The main agent, every time |
| **Handoffs** | A tool call changes **state** (`current_step` / `active_agent`); the next turn runs a different prompt/agent | Whoever is active; it can hand over |
| **Skills** | One agent **loads** specialized prompts/knowledge on demand (`load_skill`) | The single agent |
| **Router** | A classification step sends the request to one or more specialists (in parallel), then results are joined | The router code (or a classifier model) |
| **Custom workflow** | Your own LangGraph graph: deterministic steps + agent nodes | You (the graph) |

The docs compare them by **model calls** for three scenarios (Multi-agent → Performance comparison, verified 2026-10-12):

| Scenario | Subagents | Handoffs | Skills | Router |
|---|---|---|---|---|
| One-shot request ("buy coffee") | 4 | 3 | 3 | 3 |
| Repeat request (same topic, 2nd turn) | 8 | 5 | 5 | 6 |
| Multi-domain (3 domains in one question) | 5 calls, ~9K tokens | 7+ calls, ~14K+ tokens | 3 calls, ~15K tokens | 5 calls, ~9K tokens |

The pattern is clear: subagents pay an extra call per turn (the coordinator always speaks last) but keep contexts **small and isolated**; handoffs and skills **remember** who is in charge, so repeat turns are cheap; routers and subagents parallelize multi-domain questions. Part 4 measures the same thing on our desk.

---
## Part 2 · Meet the support desk

In [ ]:
# 🧪 Offline: the backend, the tools per specialist, and the 15-conversation test set.
from mastery.agents.support import SupportDesk, faq_search, load_conversations, make_support_tools

desk = SupportDesk.load()                       # customers, invoices, service status (data/samples/support-desk/)
tools = make_support_tools(desk, faq_search())  # FAQ search = BM25 over 3 Markdown files (keyless)
for domain, domain_tools in tools.items():
    print(f"{domain:<10}", [t.name for t in domain_tools])

conversations = load_conversations()
print(len(conversations), "test conversations, e.g.:")
for conv in conversations[:2] + conversations[11:14]:
    print(f"  {conv['id']}: {' | '.join(conv['turns'])[:110]}")

**You should now see** three domains with 2, 2 and 3 tools (`issue_refund` is in billing), `15 test conversations`, and five examples: S01 and S02 are one-shot refunds, S12 is a **repeat request** (two billing turns), S13 switches from billing to technical, S14 asks two domains in one message.

`issue_refund` calls `interrupt()` for refunds above **$50** (`REFUND_THRESHOLD`), so a billing manager approves them. Every tool call is logged in `desk.events`, which is how the test set checks what really happened, whatever the pattern.

---
## Part 3 · The baseline: one agent, all the tools

Before splitting anything, measure the simplest design: **one** `create_agent` with all 7 tools and one prompt. `UsageMeter` is a callback: pass it in the config and it counts every model call (and tokens) underneath, per agent name.

In [ ]:
# 🧪 Offline: the single-agent baseline on a one-shot refund (S01).
from langgraph.checkpoint.memory import InMemorySaver
from mastery.agents import UsageMeter
from mastery.agents.support import build_single_agent_baseline, offline_model, run_conversation

baseline = build_single_agent_baseline(offline_model(), tools, checkpointer=InMemorySaver())
meter = UsageMeter()
with meter:                                      # also times the block
    outcome = run_conversation(baseline, conversations[0], desk, callbacks=[meter])
print("answer :", outcome["answer"])
print("tools  :", outcome["tools"])
print("usage  :", meter.summary())
print("agents :", meter.agents())

**You should now see** `Refunded $19.00 for INV-3001 (approved by policy) ...`, `tools  : ['issue_refund']`, and `model_calls: 2` (one call to pick the tool, one to answer) with about 1,300 tokens, all under `support_agent`.

### What just happened?

`run_conversation` plays each user turn on one thread and answers any pause with a reviewer (here none: $19 is under the threshold). Two model calls is the **minimum** for any tool-using agent; every extra agent in the path adds at least one more.

---
## Part 4 · Measure every pattern on the same three scenarios

`mastery.agents.support` builds the **same** specialists (prompts + tools) in each pattern. Lessons 09.02–09.05 open each one up; here we only count.

In [ ]:
# 🧪 Offline: model calls and tokens per pattern, for the docs' three scenarios.
from mastery.agents.support import (build_handoffs_desk, build_router_desk, build_single_agent_desk,
                                    build_subagents_desk, evaluate)

BUILDERS = {
    "single agent": lambda: build_single_agent_baseline(offline_model(), tools, checkpointer=InMemorySaver()),
    "handoffs (1 agent)": lambda: build_single_agent_desk(offline_model(), tools, checkpointer=InMemorySaver()),
    "handoffs (graph)": lambda: build_handoffs_desk(offline_model, tools, checkpointer=InMemorySaver()),
    "subagents": lambda: build_subagents_desk(offline_model, tools, checkpointer=InMemorySaver()),
    "router": lambda: build_router_desk(offline_model, tools, checkpointer=InMemorySaver()),
}
SCENARIOS = {"one-shot (S01)": "S01", "repeat (S12)": "S12", "multi-domain (S14)": "S14"}
by_id = {c["id"]: c for c in conversations}

rows = []
for name, build in BUILDERS.items():
    row = {"pattern": name}
    for label, conv_id in SCENARIOS.items():
        meter = UsageMeter()
        report = evaluate(build(), [by_id[conv_id]], desk, meter=meter)
        mark = "" if report["passed"] else " ❌"
        row[label] = f"{meter.model_calls} calls / {meter.total_tokens:,} tok{mark}"
    rows.append(row)
table(rows, ["pattern", *SCENARIOS])

**You should now see** (measured, offline):

| pattern | one-shot (S01) | repeat (S12) | multi-domain (S14) |
|---|---|---|---|
| single agent | 2 calls | 4 calls | 3 calls |
| handoffs (1 agent) | 3 calls | 5 calls | 5 calls |
| handoffs (graph) | 3 calls | 5 calls | 5 calls |
| subagents | 4 calls | 8 calls | 6 calls |
| router | 2 calls | 4 calls | 4 calls |

with roughly 700–2,900 tokens per cell. The one-shot and repeat columns for subagents (4, 8) and handoffs (3, 5) match the docs' table exactly. Our router is cheaper than the docs' (2 vs 3) because its classifier is keyword code and its join is code, not model calls.

In [ ]:
# 🧪 Offline: the whole test set, per pattern (pass rate, total calls, tokens, time).
rows = []
for name, build in BUILDERS.items():
    meter = UsageMeter()
    report = evaluate(build(), conversations, desk, meter=meter)
    usage = report["usage"]
    rows.append({"pattern": name, "passed": f"{report['passed']}/{report['total']}", "model calls": usage["model_calls"],
                 "tokens": f"{usage['total_tokens']:,}", "seconds": usage["seconds"]})
table(rows, ["pattern", "passed", "model calls", "tokens", "seconds"])

**You should now see** `15/15` for every pattern, and **36 / 53 / 53 / 71 / 36** model calls (single agent / handoffs ×2 / subagents / router), with the router using about half the tokens of the others (~13K vs ~24–27K). Seconds are tiny offline (well under 1 s); with a real model each call costs ~0.3–2 s, so calls ≈ latency.

### What just happened?

On a desk this small, **the single agent wins**: same pass rate, fewest calls. That is the docs' own advice: start with one agent and add structure when something breaks. Multi-agent pays off when the single agent **stops passing**: too many tools to choose from, prompts that conflict, different teams owning parts, or contexts that grow too large. The offline model never gets confused by 7 tools; a real model with 40 tools does. So the right question is not "which pattern is best?" but "what does my eval say, and what does each fix cost?"

---
## Part 5 · Do I need multi-agent? A checklist

Go multi-agent only if at least one is true, **and** a single-agent version failed your eval:

| Signal | Pattern it points to |
|---|---|
| One agent has too many tools and picks the wrong one | **Subagents** (isolate tool sets) or **skills** (load on demand) |
| Very different instructions per domain fight each other in one prompt | **Handoffs** (one prompt active at a time) or **skills** |
| The conversation moves through stages (triage → solve → confirm) | **Handoffs** |
| Requests often span several independent domains | **Router** (parallel) or **subagents** (parallel tool calls) |
| Some steps must always happen, in order (validation, approval, logging) | **Custom workflow** (deterministic edges) |
| Specialists are built by different teams / deployed separately | **Subagents** (clear tool boundary) |

**Runtime, framework, harness** (docs: Runtimes, frameworks, and harnesses):

- **LangGraph** is the **runtime**: durable execution, streaming, HITL, persistence (Modules 07–08). Every pattern here runs on it.
- **LangChain** is the **framework**: `create_agent`, tools, middleware. Subagents, handoffs, skills are built from these pieces.
- **Deep Agents** is a **harness**: predefined tools, prompts, planning, a filesystem and built-in subagents for long, open-ended tasks. Module 11.

---
## Part 6 · Jupyter skill: long notebooks and Jupytext pairing

**Organizing.** Module 09 notebooks are long, so each is cut into **Parts** with `## Part N · ...` headings. Open *View → Table of Contents* (JupyterLab) for a clickable outline; click the blue bar left of a heading to **collapse** a whole Part. Keep one **comparison cell** per Part (like Part 4's `table(...)`) instead of scattered prints.

**Pairing.** A `.ipynb` is JSON: diffs are noisy and outputs leak into reviews. [Jupytext](https://jupytext.readthedocs.io/en/latest/paired-notebooks.html) keeps a `.py` twin in the **percent format** (`# %%` starts a code cell, `# %% [markdown]` a Markdown cell). You can review and grep the `.py`, edit either side, and `--sync` updates the older one. We try it on a **copy** of this notebook in `data/outputs/m09/` (git-ignored).

In [ ]:
# 🧪 Offline: pair a copy of this notebook with a .py file, edit the .py, sync back.
import shutil
import subprocess

import nbformat

work = REPO_ROOT / "data" / "outputs" / "m09" / "jupytext-demo"
work.mkdir(parents=True, exist_ok=True)
nb_copy = work / "overview-copy.ipynb"
shutil.copy(REPO_ROOT / "modules" / "09-multi-agent-systems" / "09-01-multi-agent-overview.ipynb", nb_copy)


def jupytext(*args):
    """Run jupytext from THIS kernel's environment (works even if it isn't on PATH)."""
    done = subprocess.run([sys.executable, "-m", "jupytext", *args], capture_output=True, text=True, cwd=work)
    return (done.stdout + done.stderr).strip().splitlines()[-1:]


print(jupytext("--set-formats", "ipynb,py:percent", nb_copy.name))
py_twin = nb_copy.with_suffix(".py")
lines = py_twin.read_text(encoding="utf-8").splitlines()
print(len(lines), "lines in", py_twin.name, "| cell markers:", sum(1 for line in lines if line.startswith("# %%")))
first_cell = next(i for i, line in enumerate(lines) if line.startswith("# %%"))
print("\n".join([*lines[:6], "# ...", *lines[first_cell:first_cell + 3]]))

**You should now see** `[jupytext] Updating overview-copy.py`, about 440 lines and **36** `# %%` cell markers (one per cell of this notebook), then the head of the `.py`: the YAML header (`formats: ipynb,py:percent`, `format_name: percent`) and the first cell, `# %% [markdown]` followed by the title as `# # 09.01 · ...` comments.

In [ ]:
# 🧪 Offline: add a cell in the .py, sync, and see it in the .ipynb.
before = len(nbformat.read(nb_copy, as_version=4).cells)
with py_twin.open("a", encoding="utf-8") as f:
    f.write('\n# %%\nprint("added in the .py file")\n')
print(jupytext("--sync", nb_copy.name))
after = nbformat.read(nb_copy, as_version=4)
print("cells:", before, "→", len(after.cells), "| last cell:", after.cells[-1].source)

**You should now see** a `[jupytext]` sync message and `cells: 36 → 37 | last cell: print("added in the .py file")`: the `.py` was newer, so `--sync` rebuilt the notebook from it (keeping the outputs it could match).

> Pairing **this course's** notebooks is optional: they are generated, and the repo commits `.ipynb` only. In your own projects, commit both files (or only the `.py`) and add `*.ipynb` diffs to code review via the `.py`. Jupytext also has a JupyterLab command (*Pair Notebook with percent Script*) that does the same as `--set-formats`.

---
## Part 7 · 💸 The baseline with a real model

> 💸 Calls `gpt-oss-20b` several times (multi-agent = many small calls), a few thousand tokens of your free-tier quota. Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 💸 LIVE: the single-agent baseline with gpt-oss-20b on three conversations (S01, S06, S09).
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    llm = get_model("fast", temperature=0, reasoning_effort="low")
    live = build_single_agent_baseline(llm, tools, checkpointer=InMemorySaver())
    meter = UsageMeter()
    report = evaluate(live, [by_id[i] for i in ("S01", "S06", "S09")], desk, meter=meter)
    for row in report["rows"]:
        print(row["id"], "✅" if row["passed"] else "❌", row["outcome"]["tools"], "|", row["outcome"]["answer"][:120])
    print(f"passed {report['passed']}/{report['total']} |", report["usage"])
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) three rows; usually all ✅ with the right tool each (`issue_refund`, `search_faq`, `send_password_reset`), about 2 model calls per conversation and a few thousand tokens in total (real tool schemas are bigger than the offline estimate). A ❌ shows which check failed in `report["rows"]`; read its answer. Without a key: `⏭️ Skipped`.

---
## ⚠️ Pitfalls and troubleshooting (Windows Git Bash included)

| Symptom | Cause | Fix |
|---|---|---|
| `ImportError: cannot import name 'UsageMeter' from 'mastery.agents'` | An old `src/mastery/agents.py` is imported, or `pip install -e` wasn't re-run | Delete `src/mastery/agents.py`, re-run `python -m pip install -e ".[dev,embeddings]"`, restart the kernel |
| `No module named jupytext` | Module 09 install not done in this env | `python -m pip install -e ".[dev]"` (adds the `[notebooks]` extra), restart the kernel |
| `jupytext: command not found` in Git Bash | The env's `Scripts/` not on `PATH` | Use `python -m jupytext ...` (what Part 6 does) |
| Git warns `LF will be replaced by CRLF` for a paired `.py` | `core.autocrlf=true` on Windows | Harmless; or add `*.py text eol=lf` to `.gitattributes` |
| Counts differ from the table | You changed the test set or tools | Re-run Part 4; the numbers are measured, not hard-coded |
| LIVE run is slow, `429` in logs | Free-tier requests-per-minute limit; multi-agent makes many calls | Wait; `get_model` retries. Run fewer conversations |

## ⚠️ Production notes

> ⚠️ **Production note: start with one agent and an eval.**
> Build the single-agent version first, measure it on a test set, and add agents only for the failures you can name. Keep the baseline in your comparison table forever: it's the bar every new design has to beat.

> ⚠️ **Production note: count calls, not agents.**
> Latency and cost scale with **model calls on the critical path**. A diagram with 5 agents can be cheaper than one with 3 if most branches don't run. Measure per scenario (one-shot, repeat, multi-domain), not just in total.

## 🏋️ Exercises

1. **(Easy)** Add a fourth scenario to `SCENARIOS`: the domain switch S13. Which pattern pays most for switching, and why?
2. **(Medium)** Print `meter.agents()` for the subagents desk on S14. Which agent makes the most calls, and what does that tell you about where to spend a stronger (pricier) model?
3. **(Medium)** Write your own 16th conversation (an account question with an email change) as a dict like the JSONL lines, and run `evaluate` on it for two patterns.
4. **(Stretch)** Pair this notebook (a copy) with `py:light` instead of `py:percent` and compare the two `.py` files. Which one would you rather review?

<details>
<summary><b>Show solutions</b></summary>

**1.** `SCENARIOS["switch (S13)"] = "S13"` and re-run. Measured: handoffs pay a **transfer** (the billing agent hands over to technical: an extra call), subagents pay the coordinator's two calls per turn; the single agent just calls another tool.

**2.**
```python
meter = UsageMeter()
evaluate(BUILDERS["subagents"](), [by_id["S14"]], desk, meter=meter)
print(meter.agents())   # coordinator: 2 calls; billing and technical: 2 each
```
The coordinator is called on **every** turn, so it dominates in longer conversations: that's where a better model changes quality most, and where a cheaper one saves most.

**3.**
```python
mine = {"id": "X01", "turns": ["I'm C-1002, please change my email to ben@new.example"],
        "expect": {"tools": ["update_email"], "interrupts": 0, "contains": ["confirmation"]}}
for name in ("single agent", "subagents"):
    print(name, evaluate(BUILDERS[name](), [mine], desk)["passed"])
```

**4.** `jupytext("--set-formats", "ipynb,py:light", nb_copy.name)` on a fresh copy. The light format marks only Markdown as comments and separates code cells by blank lines; percent has an explicit `# %%` per cell, which survives editing better and is understood by VS Code and Spyder.
</details>

## 🔎 Docs drill (5 min)

The docs' multi-agent overview compares patterns on a few criteria. Which ones, and which pattern does it suggest when you are not sure you need multiple agents?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

Open [Multi-agent](https://docs.langchain.com/oss/python/langchain/multi-agent): find the comparison of the patterns (who controls the flow, context isolation, model calls per turn) and the advice to start with a single agent and its tools, adding agents only when the measurements say so, as Part 5 here did.

</details>

## ✅ Checkpoint

- [ ] I can explain who controls the flow in each of the five patterns.
- [ ] I measured model calls and tokens per pattern and can say why subagents cost more per turn.
- [ ] I can argue when a single agent is the better design.
- [ ] I paired a notebook copy with Jupytext and synced an edit back.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Read the [Multi-agent](https://docs.langchain.com/oss/python/langchain/multi-agent/index) overview's **Performance comparison** and note in [`docs/research-log.md`](../../docs/research-log.md): the numbers in its tables today vs the ones in Part 1 (verified 2026-10-12), and whether the list of patterns changed.

## Summary

- Five patterns: subagents, handoffs, skills, router, custom workflow. The difference is **who decides what runs next**.
- Measured on one desk: subagents 4 calls one-shot / 8 for a repeat request; handoffs 3 / 5 (same as the docs); the single agent is cheapest when it works.
- Add agents for named failures (tool confusion, conflicting prompts, stages, parallel domains, mandatory steps), and keep the baseline in the comparison.
- LangGraph = runtime, LangChain = framework, Deep Agents = harness (Module 11).
- Long notebooks: Parts + TOC + one comparison cell per Part; Jupytext pairs `.ipynb` with a reviewable `.py`.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 09.02 · Subagents (supervisor-style orchestration)** → [`09-02-subagents.ipynb`](09-02-subagents.ipynb).

[Module 09 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)